In [1]:
from pathlib import Path
import re
import json
from collections import Counter

TRACE_DIR = Path("traces")
PROMPT_DIR = Path("prompts")
PROMPT_DIR.mkdir(parents=True, exist_ok=True)

OUT_JSONL = Path("satp_all_path_summaries.jsonl")
OUT_TXT = Path("satp_all_path_summaries.txt")

HEX_RE = re.compile(r"#x([0-9a-fA-F]+)")

def norm_hex(x):
    if x is None:
        return None
    x = x.lower()[-16:]
    return x.rjust(16, "0")

def extract_hex(line):
    m = HEX_RE.search(line)
    return norm_hex(m.group(1)) if m else None

def split_trace_blocks(text):
    lines = text.splitlines()
    starts = [i for i, line in enumerate(lines) if line.strip().startswith("(trace")]
    if not starts:
        return [text] if text.strip() else []

    blocks = []
    for i, start in enumerate(starts):
        end = starts[i + 1] if i + 1 < len(starts) else len(lines)
        block = "\n".join(lines[start:end]).strip()
        if block:
            blocks.append(block)
    return blocks

def is_tool_error(text):
    low = text.lower()
    return any(x in low for x in ["panicked", "panic", "dumped core", "aborted", "timeout"])

def has_mcause_2(block):
    return bool(re.search(r"write-reg \|mcause\|.*#x0*2\)", block, re.S))

def exception_name(block):
    if has_mcause_2(block):
        return "illegal instruction"
    if "write-reg |mcause|" in block:
        return "exception"
    return "none"

def important_events(block):
    keywords = [
        "assume-reg |cur_privilege|",
        "assume-reg |satp|",
        "read-reg |cur_privilege|",
        "write-reg |cur_privilege|",
        "read-reg |mstatus|",
        "write-reg |mstatus|",
        "read-reg |satp|",
        "write-reg |satp|",
        "read-reg |misa|",
        "read-reg |x10|",
        "read-reg |x11|",
        "write-reg |x10|",
        "write-reg |x11|",
        "read-reg |mcause|",
        "write-reg |mcause|",
        "write-reg |mtval|",
        "write-reg |mepc|",
        "write-reg |nextPC|",
        "SATPMode",
        "Retired",
        "panicked",
        "dumped core",
    ]
    out = []
    for line in block.splitlines():
        s = line.strip()
        if any(k in s for k in keywords):
            out.append(s)
    return out[:120]

def csr_priv_from_name(name):
    m = re.match(r"csr__(Machine|Supervisor|User)__", name)
    return m.group(1) if m else "Unknown"

def csr_instr_from_name(name):
    name = name.removesuffix(".trace")
    name = re.sub(r"^csr__(Machine|Supervisor|User)__", "", name)
    parts = name.split("_")

    if parts[:3] == ["csrr", "x10", "satp"]:
        return "csrr x10, satp"
    if parts[:3] == ["csrw", "satp", "x10"]:
        return "csrw satp, x10"
    if parts[:4] == ["csrrw", "x10", "satp", "x11"]:
        return "csrrw x10, satp, x11"
    if parts[:4] == ["csrrs", "x10", "satp", "x11"]:
        return "csrrs x10, satp, x11"
    if parts[:4] == ["csrrc", "x10", "satp", "x11"]:
        return "csrrc x10, satp, x11"
    if parts[:4] == ["csrrwi", "x10", "satp", "0"]:
        return "csrrwi x10, satp, 0"
    if parts[:4] == ["csrrwi", "x10", "satp", "1"]:
        return "csrrwi x10, satp, 1"
    if parts[:4] == ["csrrsi", "x10", "satp", "1"]:
        return "csrrsi x10, satp, 1"
    if parts[:4] == ["csrrci", "x10", "satp", "1"]:
        return "csrrci x10, satp, 1"

    return name.replace("_", " ")

def instruction_kind(instr):
    low = instr.lower()
    if low.startswith("csrr "):
        return "read"
    if low.startswith("csrw "):
        return "write"
    if low.startswith("csrrw "):
        return "read_write"
    if low.startswith("csrrs ") or low.startswith("csrrc "):
        return "read_modify_write"
    if low.startswith("csrrwi "):
        return "read_write_immediate"
    if low.startswith("csrrsi ") or low.startswith("csrrci "):
        return "read_modify_write_immediate"
    return "unknown"

def parse_csr_trace(path):
    text = path.read_text(errors="replace")
    blocks = split_trace_blocks(text)
    rows = []

    for idx, block in enumerate(blocks):
        priv = csr_priv_from_name(path.name)
        instr = csr_instr_from_name(path.name)
        exc = exception_name(block)

        reads_satp = "read-reg |satp|" in block
        writes_satp = "write-reg |satp|" in block
        reads_mstatus = "read-reg |mstatus|" in block
        reads_x11 = "read-reg |x11|" in block
        writes_x10 = "write-reg |x10|" in block
        tool_error = is_tool_error(block)

        if tool_error:
            path_name = "CSR access tool error"
            summary = "Isla/Sailor failed before producing a usable CSR-access architectural trace."
            status = "tool_error"
        elif exc == "illegal instruction":
            status = "trapped"
            if priv == "User":
                path_name = "User-mode SATP access illegal"
                summary = "SATP CSR access from User mode raises an illegal instruction exception; SATP is not read or written normally."
            elif priv == "Supervisor" and reads_mstatus:
                path_name = "Supervisor-mode SATP access blocked by TVM-related path"
                summary = "Supervisor-mode SATP access can raise an illegal instruction exception when the SATP accessibility check fails through mstatus.TVM."
            else:
                path_name = f"{priv}-mode SATP access illegal"
                summary = f"{priv}-mode SATP access raises an illegal instruction exception."
        else:
            status = "success"
            if reads_satp and writes_satp and writes_x10:
                path_name = f"{priv}-mode SATP read/write succeeds"
                summary = f"{priv}-mode SATP CSR operation succeeds: SATP is read, SATP may be written, and the old SATP value is written to x10."
            elif writes_satp:
                path_name = f"{priv}-mode SATP write succeeds"
                summary = f"{priv}-mode SATP write succeeds and SATP is updated."
            elif reads_satp and writes_x10:
                path_name = f"{priv}-mode SATP read succeeds"
                summary = f"{priv}-mode SATP read succeeds and writes the SATP value to x10."
            else:
                path_name = f"{priv}-mode SATP path succeeds"
                summary = f"{priv}-mode SATP-related path completes without an exception."

        rows.append({
            "experiment": "csr_access",
            "path_id": f"{path.stem}__path_{idx:03d}",
            "trace_file": str(path),
            "path_name": path_name,
            "privilege": priv,
            "instruction": instr,
            "instruction_kind": instruction_kind(instr),
            "exception": exc,
            "status": status,
            "tool_error": tool_error,
            "reads_satp": reads_satp,
            "writes_satp": writes_satp,
            "reads_mstatus": reads_mstatus,
            "reads_x11": reads_x11,
            "writes_x10": writes_x10,
            "summary": summary,
            "important_trace_events": important_events(block),
        })

    return rows

def mode_nibble(hex64):
    return hex64[0].lower() if hex64 else None

def mode_name(n):
    return {
        "0": "Bare",
        "8": "Sv39",
        "9": "Sv48",
        "a": "Sv57",
        "b": "Sv64-reserved",
        "c": "reserved/custom-range",
        "d": "reserved/custom-range",
        "e": "custom",
        "f": "custom",
    }.get(n, "unsupported/reserved")

def legalize_written_value(name):
    m = re.search(r"_x10_([0-9a-fA-F]{16})\.trace$", name)
    return norm_hex(m.group(1)) if m else None

def legalize_instruction(name):
    if "csrrw_x11_satp_x10" in name:
        return "csrrw x11, satp, x10"
    if "csrw_satp_x10" in name:
        return "csrw satp, x10"
    return "unknown"

def parse_legalize_trace(path):
    text = path.read_text(errors="replace")
    lines = text.splitlines()

    written = legalize_written_value(path.name)
    instr = legalize_instruction(path.name)
    tool_error = is_tool_error(text)

    read_x10 = []
    read_satp = []
    write_satp = []
    readback_after_write = []
    write_x11 = []

    seen_write = False
    for line in lines:
        if "read-reg |x10|" in line:
            read_x10.append(extract_hex(line))
        elif "read-reg |satp|" in line:
            val = extract_hex(line)
            read_satp.append(val)
            if seen_write:
                readback_after_write.append(val)
        elif "write-reg |satp|" in line:
            write_satp.append(extract_hex(line))
            seen_write = True
        elif "write-reg |x11|" in line:
            write_x11.append(extract_hex(line))

    old_satp = read_satp[0] if read_satp else None
    final_write = write_satp[-1] if write_satp else None
    final_readback = readback_after_write[-1] if readback_after_write else None

    if tool_error:
        status = "tool_error"
        path_name = "SATP legalize tool error"
        summary = f"For {instr}, writing 0x{written} causes an Isla/Sailor tool error before a usable architectural trace is produced."
    elif final_write is None:
        status = "no_satp_write"
        path_name = "SATP legalize no write observed"
        summary = f"For {instr}, writing 0x{written} does not produce an observed SATP write in the trace."
    elif final_write == written:
        status = "accepted"
        path_name = "SATP write accepted"
        summary = f"For {instr}, writing 0x{written} to satp is accepted and reflected in SATP."
    elif old_satp is not None and final_write == old_satp:
        status = "ignored_or_preserved_old"
        path_name = "SATP write ignored or preserves old value"
        summary = f"For {instr}, writing 0x{written} to satp preserves the old SATP value 0x{old_satp}."
    else:
        status = "modified"
        path_name = "SATP write modified"
        summary = f"For {instr}, writing 0x{written} to satp changes SATP to 0x{final_write}, different from both old and written values."

    return {
        "experiment": "legalize_write_readback",
        "path_id": path.stem,
        "trace_file": str(path),
        "path_name": path_name,
        "privilege": "Machine",
        "instruction": instr,
        "instruction_kind": instruction_kind(instr),
        "written_value": written,
        "mode_nibble": mode_nibble(written),
        "mode_name_hint": mode_name(mode_nibble(written)),
        "old_satp": old_satp,
        "final_write_satp": final_write,
        "final_readback_satp": final_readback,
        "write_satp_values": write_satp,
        "write_x11_values": write_x11,
        "exception": "none" if not tool_error else "tool_error",
        "status": status,
        "tool_error": tool_error,
        "summary": summary,
        "important_trace_events": important_events(text),
    }

rows = []

for trace_file in sorted(TRACE_DIR.glob("csr__*.trace")):
    rows.extend(parse_csr_trace(trace_file))

for trace_file in sorted(TRACE_DIR.glob("legalize__*.trace")):
    rows.append(parse_legalize_trace(trace_file))

with OUT_JSONL.open("w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")

lines = []

for i, row in enumerate(rows, 1):
    lines.append(f"Path {i}: {row['path_name']}")
    lines.append(f"Path ID: {row['path_id']}")
    lines.append(f"Experiment: {row['experiment']}")
    lines.append(f"Trace file: {row['trace_file']}")
    lines.append("")
    lines.append("Condition:")
    lines.append(f"- Privilege: {row.get('privilege')}.")
    lines.append(f"- Instruction: {row.get('instruction')}.")
    lines.append(f"- Instruction kind: {row.get('instruction_kind')}.")

    if row["experiment"] == "legalize_write_readback":
        lines.append(f"- Written value: 0x{row.get('written_value')}.")
        lines.append(f"- MODE nibble: {row.get('mode_nibble')} ({row.get('mode_name_hint')}).")

    lines.append("")
    lines.append("Outcome:")
    lines.append(f"- Status: {row.get('status')}.")
    lines.append(f"- Exception: {row.get('exception')}.")
    lines.append(f"- Tool error: {row.get('tool_error')}.")

    if row["experiment"] == "csr_access":
        lines.append(f"- Reads mstatus: {row.get('reads_mstatus')}.")
        lines.append(f"- Reads satp: {row.get('reads_satp')}.")
        lines.append(f"- Writes satp: {row.get('writes_satp')}.")
        lines.append(f"- Writes x10: {row.get('writes_x10')}.")
    else:
        lines.append(f"- Old satp: {('0x' + row['old_satp']) if row.get('old_satp') else 'unknown'}.")
        lines.append(f"- Final write satp: {('0x' + row['final_write_satp']) if row.get('final_write_satp') else 'none'}.")
        lines.append(f"- Final readback satp: {('0x' + row['final_readback_satp']) if row.get('final_readback_satp') else 'none'}.")

    lines.append("")
    lines.append("Summary:")
    lines.append(row["summary"])
    lines.append("")
    lines.append("Important trace events:")
    for e in row.get("important_trace_events", []):
        lines.append(f"- {e}")
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

OUT_TXT.write_text("\n".join(lines), encoding="utf-8")

for row in rows:
    prompt = f"""You are given one Sail/Isla trace-derived path for RISC-V SATP.

Task:
Convert this path into exactly one atomic architectural rule about SATP.

Important:
- Use only this path summary and trace events.
- Do not use emulator behavior.
- Do not use known bugs.
- Do not infer unrelated behavior.
- If tool_error is true, do not infer an ISA rule; output RULE: TOOL_ERROR with the reason.
- The rule must describe one condition -> one architectural outcome.
- Output only one line beginning with RULE:.

Path:
{json.dumps({k: v for k, v in row.items() if k != "important_trace_events"}, indent=2)}

Important trace events:
{chr(10).join(row.get("important_trace_events", []))}
"""
    (PROMPT_DIR / f"{row['path_id']}.txt").write_text(prompt, encoding="utf-8")

print("total rows:", len(rows))
print("by experiment:", Counter(r["experiment"] for r in rows))
print("by status:", Counter(r["status"] for r in rows))
print("by exception:", Counter(r["exception"] for r in rows))
print("tool errors:", sum(r["tool_error"] for r in rows))
print("wrote:", OUT_JSONL)
print("wrote:", OUT_TXT)
print("wrote prompts:", PROMPT_DIR)

total rows: 86
by experiment: Counter({'csr_access': 54, 'legalize_write_readback': 32})
by status: Counter({'success': 36, 'trapped': 18, 'accepted': 16, 'tool_error': 10, 'ignored_or_preserved_old': 6})
by exception: Counter({'none': 58, 'illegal instruction': 18, 'tool_error': 10})
tool errors: 10
wrote: satp_all_path_summaries.jsonl
wrote: satp_all_path_summaries.txt
wrote prompts: prompts


In [4]:
from pathlib import Path
import re
import json
from collections import Counter

TRACE_DIR = Path("traces")
OLD_JSONL = Path("satp_all_path_summaries.jsonl")

OUT_JSONL = Path("satp_all_path_summaries_with_translation.jsonl")
OUT_TXT = Path("satp_all_path_summaries_with_translation.txt")

def split_trace_blocks(text):
    lines = text.splitlines()
    starts = [i for i, line in enumerate(lines) if line.strip().startswith("(trace")]
    if not starts:
        return [text] if text.strip() else []
    blocks = []
    for i, start in enumerate(starts):
        end = starts[i + 1] if i + 1 < len(starts) else len(lines)
        block = "\n".join(lines[start:end]).strip()
        if block:
            blocks.append(block)
    return blocks

def is_tool_error(text):
    low = text.lower()
    return any(x in low for x in ["panicked", "panic", "dumped core", "aborted", "timeout", "error:"])

def translation_priv_from_name(name):
    m = re.match(r"translation__(Machine|Supervisor|User)__", name)
    return m.group(1) if m else "Unknown"

def translation_instr_from_name(name):
    if "__ld_x5_0_x6" in name:
        return "ld x5, 0(x6)"
    if "__sd_x5_0_x6" in name:
        return "sd x5, 0(x6)"
    return "unknown"

def instruction_kind(instr):
    if instr.startswith("ld "):
        return "load"
    if instr.startswith("sd "):
        return "store"
    return "unknown"

def extract_pa(block):
    m = re.search(r"\(\|pa\| #x([0-9a-fA-F]+)\)", block)
    return m.group(1).lower() if m else None

def important_translation_events(block):
    keywords = [
        "assume-reg |cur_privilege|",
        "assume-reg |satp|",
        "assume-reg |x5|",
        "assume-reg |x6|",
        "read-reg |cur_privilege|",
        "read-reg |mstatus|",
        "read-reg |satp|",
        "read-reg |x5|",
        "read-reg |x6|",
        "write-reg |x5|",
        "read-mem",
        "write-mem",
        "SATPMode",
        "Retired",
        "mcause",
        "panicked",
        "dumped core",
        "error:",
    ]
    out = []
    for line in block.splitlines():
        s = line.strip()
        if any(k in s for k in keywords):
            out.append(s)
    return out[:120]

def parse_translation_trace(path):
    text = path.read_text(errors="replace")
    blocks = split_trace_blocks(text)
    rows = []

    priv = translation_priv_from_name(path.name)
    instr = translation_instr_from_name(path.name)
    kind = instruction_kind(instr)

    for idx, block in enumerate(blocks):
        tool_error = is_tool_error(block)
        reads_satp = "read-reg |satp|" in block
        reads_mstatus = "read-reg |mstatus|" in block
        reads_mem = "read-mem" in block
        writes_mem = "write-mem" in block
        writes_x5 = "write-reg |x5|" in block
        retired_success = "RETIRE_SUCCESS" in block
        pa = extract_pa(block)

        if tool_error:
            status = "tool_error"
            exception = "tool_error"
            path_name = "Bare translation tool error"
            summary = f"{priv}-mode Bare translation for {instr} hit a Sail/Isla tool error."
        elif retired_success and (reads_mem or writes_mem):
            status = "success"
            exception = "none"
            path_name = f"{priv}-mode Bare translation {kind} succeeds"
            if kind == "load":
                summary = (
                    f"{priv}-mode load with satp=0 uses Bare translation: "
                    f"the access goes to physical address 0x{pa}, then the loaded value is written to x5."
                )
            else:
                summary = (
                    f"{priv}-mode store with satp=0 uses Bare translation: "
                    f"the value in x5 is written to physical address 0x{pa}."
                )
        else:
            status = "unknown"
            exception = "unknown"
            path_name = f"{priv}-mode Bare translation unclear"
            summary = f"{priv}-mode {instr} produced a trace, but the memory outcome was not clearly parsed."

        rows.append({
            "experiment": "translation_bare",
            "path_id": f"{path.stem}__path_{idx:03d}",
            "trace_file": str(path),
            "path_name": path_name,
            "privilege": priv,
            "instruction": instr,
            "instruction_kind": kind,
            "satp_value": "0000000000000000",
            "translation_mode": "Bare",
            "physical_address": pa,
            "reads_satp": reads_satp,
            "reads_mstatus": reads_mstatus,
            "reads_mem": reads_mem,
            "writes_mem": writes_mem,
            "writes_x5": writes_x5,
            "status": status,
            "exception": exception,
            "tool_error": tool_error,
            "summary": summary,
            "important_trace_events": important_translation_events(block),
        })

    return rows

old_rows = []
with OLD_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        old_rows.append(json.loads(line))

translation_rows = []
translation_trace_files = sorted(TRACE_DIR.glob("translation__*.trace"))

for trace_file in translation_trace_files:
    translation_rows.extend(parse_translation_trace(trace_file))

all_rows = old_rows + translation_rows

with OUT_JSONL.open("w", encoding="utf-8") as f:
    for row in all_rows:
        f.write(json.dumps(row) + "\n")

lines = []
for i, row in enumerate(all_rows, 1):
    lines.append(f"Path {i}: {row.get('path_name')}")
    lines.append(f"Path ID: {row.get('path_id')}")
    lines.append(f"Experiment: {row.get('experiment')}")
    lines.append(f"Trace file: {row.get('trace_file')}")
    lines.append("")
    lines.append("Condition:")
    lines.append(f"- Privilege: {row.get('privilege')}.")
    lines.append(f"- Instruction: {row.get('instruction')}.")
    lines.append(f"- Instruction kind: {row.get('instruction_kind')}.")
    if row.get("experiment") == "translation_bare":
        lines.append(f"- satp value: 0x{row.get('satp_value')}.")
        lines.append(f"- Translation mode: {row.get('translation_mode')}.")
    lines.append("")
    lines.append("Outcome:")
    lines.append(f"- Status: {row.get('status')}.")
    lines.append(f"- Exception: {row.get('exception')}.")
    lines.append(f"- Tool error: {row.get('tool_error')}.")
    if row.get("experiment") == "translation_bare":
        lines.append(f"- Physical address: {('0x' + row['physical_address']) if row.get('physical_address') else 'unknown'}.")
        lines.append(f"- Reads memory: {row.get('reads_mem')}.")
        lines.append(f"- Writes memory: {row.get('writes_mem')}.")
        lines.append(f"- Writes x5: {row.get('writes_x5')}.")
    lines.append("")
    lines.append("Summary:")
    lines.append(row.get("summary", ""))
    lines.append("")
    lines.append("Important trace events:")
    for e in row.get("important_trace_events", []):
        lines.append(f"- {e}")
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

OUT_TXT.write_text("\n".join(lines), encoding="utf-8")

print("old rows:", len(old_rows))
print("translation trace files:", len(translation_trace_files))
print("translation rows:", len(translation_rows))
print("total rows:", len(all_rows))
print("by experiment:", Counter(r["experiment"] for r in all_rows))
print("translation status:", Counter(r["status"] for r in translation_rows))
print("wrote:", OUT_JSONL)
print("wrote:", OUT_TXT)

old rows: 86
translation trace files: 6
translation rows: 12
total rows: 98
by experiment: Counter({'csr_access': 54, 'legalize_write_readback': 32, 'translation_bare': 12})
translation status: Counter({'success': 12})
wrote: satp_all_path_summaries_with_translation.jsonl
wrote: satp_all_path_summaries_with_translation.txt


In [5]:
from pathlib import Path
import re

INPUT_FILE = Path("rule.txt")
OUTPUT_FILE = Path("rule_clean.txt")

text = INPUT_FILE.read_text(errors="ignore")

rule_pattern = re.compile(
    r"(?m)^RULE(?:\s+Path)?\s+\d+\s*:\s*"
)

matches = list(rule_pattern.finditer(text))

if not matches:
    raise ValueError("No RULE lines found in rule.txt")

blocks = []

for i, m in enumerate(matches):
    start = m.start()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
    block = text[start:end].strip()
    blocks.append(block)

seen = set()
clean_rules = []

for block in blocks:
    body = re.sub(r"^RULE(?:\s+Path)?\s+\d+\s*:\s*", "", block, flags=re.IGNORECASE).strip()

    key = re.sub(r"\s+", " ", body).strip().lower()

    if key in seen:
        continue

    seen.add(key)
    clean_rules.append(body)

output_lines = []

for i, body in enumerate(clean_rules, 1):
    output_lines.append(f"RULE {i}: {body}")

OUTPUT_FILE.write_text("\n".join(output_lines) + "\n")

print("input rules:", len(blocks))
print("unique rules:", len(clean_rules))
print("removed duplicates:", len(blocks) - len(clean_rules))
print("wrote:", OUTPUT_FILE)

input rules: 98
unique rules: 66
removed duplicates: 32
wrote: rule_clean.txt
